# Day 16 · Collaborative map — Something in the Way

**Question.** What blocks Chicago's sidewalks, where, and when in the year? The map is made with Chicago
residents: every block's color comes from the 311 reports they sent about something in the way.

**Approach.** Three 311 request types: scooters parked in the way (E-Scooter Parking Complaint), Divvy
bikes parked in the way (Divvy Bike Parking Complaint) and sidewalks not cleared of snow (Snow – Uncleared
Sidewalk Complaint), January 2022 to October 2026. Each half-mile block of the grid is shaded on two
axes: how many reports it got, and what share of them came in winter (December to February). Two charts
give every month of every year, one for wheels and one for snow.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, settings |
| 2. Pull | 311 requests of five types (dated snapshot), community areas |
| 3. Validate | Types, dates, duplicates, coordinates |
| 4. Transform | Window, kinds, grid miles, half-mile blocks |
| 5. Analyze | Blocks and their classes, months by year, community areas |
| 6. QA | Seasons, class sizes, a cross-check with the portal |
| 7. Export | `plate.json` |

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

from atlas import socrata
from atlas.grid import miles_from_origin
from atlas.ledger import Ledger, write_plate

logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger('day16')
DAY = Path('.')
REPO = DAY.resolve().parents[1]
RAW = DAY / 'data' / 'raw'
OUT = DAY / 'output'
for p in (RAW, OUT):
    p.mkdir(parents=True, exist_ok=True)

DATASET = 'v6vf-nfxy'                       # 311 Service Requests
TYPES = {
    'E-Scooter Parking Complaint': 'scooter',
    'Divvy Bike Parking Complaint': 'divvy',
    'Snow – Uncleared Sidewalk Complaint': 'snow',
    'Snow - Object/Dibs Removal Request': 'dibs',          # a note: chairs holding a shoveled parking spot
    'Submerged Divvy or Lime Device in the Lake/River': 'sunk',  # a note
}
WHEELS = ('scooter', 'divvy')
START, END = '2022-01-01', '2026-11-01'     # Jan 2022 - Oct 2026: complete months; 311 took its first scooter report in Apr 2022
LAST12 = '2025-11-01'                       # the last 12 months of the window
YEARS = list(range(2022, 2027))
STEP = 0.5                                  # miles: half-mile blocks of the grid (an arterial every half mile)
REPORT_BREAKS = [15, 45]                    # reports per block: under 15, 15-44, 45 or more
WINTER = (12, 1, 2)                         # December to February
WINTER_BREAKS = [1 / 3, 2 / 3]              # share in winter: under a third, a third to two thirds, over two thirds

## 2. Pull
One dated snapshot of the five types, all years, saved gzipped (the portal adds rows daily).

In [ ]:
cols = ['sr_number', 'sr_type', 'created_date', 'latitude', 'longitude', 'community_area', 'ward', 'origin', 'duplicate', 'legacy_record']
existing = sorted(RAW.glob('sr311_obstacles_*.csv.gz'))
if existing:
    path = existing[-1]
    sr = pd.read_csv(path, dtype=str)
else:
    where = 'sr_type in(' + ','.join(f"'{t}'" for t in TYPES) + ')'
    sr = socrata.fetch(DATASET, select=','.join(cols), where=where, order='sr_number')
    path = RAW / f'sr311_obstacles_{date.today().isoformat()}.csv.gz'
    sr[cols].to_csv(path, index=False)
    sr = pd.read_csv(path, dtype=str)
pulled = date.fromisoformat(path.name.split('_')[-1][:10])
log.info('%s: %d rows, pulled %s', path.name, len(sr), pulled)
areas = json.load(open(REPO / 'days' / 'day04-clusters' / 'data' / 'raw' / 'community_areas_2026-11-04.geojson'))

## 3. Validate

In [ ]:
assert set(sr['sr_type']) == set(TYPES), set(sr['sr_type']) ^ set(TYPES)
assert sr['sr_number'].is_unique
sr['t'] = pd.to_datetime(sr['created_date'])
log.info('first %s, last %s', sr['t'].min(), sr['t'].max())
assert sr['t'].max() >= pd.Timestamp(END), 'the snapshot must cover the whole window'
dup = sr['duplicate'].str.lower().eq('true')
geo = sr['latitude'].notna() & sr['longitude'].notna()
log.info('duplicates %d (%.0f%%), no coordinates %d', dup.sum(), 100 * dup.mean(), (~geo).sum())
sr['kind'] = sr['sr_type'].map(TYPES)
scooter_first = sr.loc[sr['kind'] == 'scooter', 't'].min()
log.info('first scooter report %s', scooter_first)
pd.crosstab(sr['kind'], sr['t'].dt.year)

## 4. Transform
Duplicates (flagged by the city) and rows without coordinates are dropped; positions in grid miles; each report falls in one half-mile block (named by its south-west corner).

In [ ]:
inwin = (sr['t'] >= START) & (sr['t'] < END)
win = sr[~dup & geo & inwin].copy()
win['x'], win['y'] = miles_from_origin(win['latitude'].astype(float).to_numpy(), win['longitude'].astype(float).to_numpy())
win['m'] = win['t'].dt.month
win['yr'] = win['t'].dt.year
win['bx'] = np.floor(win['x'] / STEP) * STEP
win['by'] = np.floor(win['y'] / STEP) * STEP
win['ca'] = pd.to_numeric(win['community_area'], errors='coerce')
names = {int(f['properties']['area_numbe']): f['properties']['community'].title() for f in areas['features']}
win['area'] = win['ca'].map(names)
win['group'] = np.where(win['kind'].isin(WHEELS), 'wheels', win['kind'])
log.info('in window: %d reports, %s', len(win), win['kind'].value_counts().to_dict())
main = win[win['group'].isin(['wheels', 'snow'])]

## 5. Analyze

In [ ]:
cls = lambda v, br: int(np.searchsorted(br, v, side='right'))
rows = []
for (bx, by), g in main.groupby(['bx', 'by']):
    n = g['kind'].value_counts()
    tot = len(g)
    ws = float(g['m'].isin(WINTER).mean())
    rows.append(dict(x=round(float(bx), 1), y=round(float(by), 1), scooter=int(n.get('scooter', 0)), divvy=int(n.get('divvy', 0)),
                     snow=int(n.get('snow', 0)), n=tot, winter=round(ws, 3), peak=int(g['m'].value_counts().idxmax()),
                     area=g['area'].value_counts().index[0] if g['area'].notna().any() else '',
                     tc=cls(tot, REPORT_BREAKS), wc=cls(ws, WINTER_BREAKS)))
blocks = pd.DataFrame(rows)
grid = pd.crosstab(blocks['tc'], blocks['wc']).reindex(index=range(3), columns=range(3), fill_value=0)

# months by year, citywide
monthly = {grp: {str(yr): main[(main['group'] == grp) & (main['yr'] == yr)].groupby('m').size().reindex(range(1, 13), fill_value=0).tolist()
                 for yr in YEARS} for grp in ('wheels', 'snow')}
year_tot = {grp: {str(yr): int(sum(v)) for yr, v in monthly[grp].items()} for grp in monthly}

last = main[main['t'] >= LAST12]
g12 = last['group'].value_counts()
notes = win[win['kind'].isin(['dibs', 'sunk'])]['kind'].value_counts()

# the darkest corners of the key: the busiest blocks, mostly Mar-Nov (wheels) or mostly Dec-Feb (snow)
def corner(wc):
    b = blocks[(blocks['tc'] == 2) & (blocks['wc'] == wc)]
    return dict(blocks=int(len(b)), reports=int(b['n'].sum()), areas=b.groupby('area')['n'].sum().sort_values(ascending=False).index[:4].tolist())
warm, cold = corner(0), corner(2)
quiet = blocks[blocks['tc'] == 0]
ca = main[main['group'] == 'wheels'].groupby('area').size().sort_values(ascending=False)

summary = dict(
    window=[START, END], last12=[LAST12, END], years=YEARS, scooterFirst=scooter_first.date().isoformat(),
    reports=int(len(main)), wheels=int((main['group'] == 'wheels').sum()), snow=int((main['group'] == 'snow').sum()),
    scooter=int((main['kind'] == 'scooter').sum()), divvy=int((main['kind'] == 'divvy').sum()),
    wheels12=int(g12.get('wheels', 0)), snow12=int(g12.get('snow', 0)),
    yearTotals=year_tot,
    blocks=int(len(blocks)), grid=grid.values.tolist(), warm=warm, cold=cold,
    quietBlocks=int(len(quiet)), quietShareOfBlocks=float(len(quiet) / len(blocks)),
    topArea=ca.index[0], topAreaShare=float(ca.iloc[0] / ca.sum()), top10Share=float(ca.iloc[:10].sum() / ca.sum()),
    snowWinterShare=float(main[(main['group'] == 'snow') & main['m'].isin(WINTER)].shape[0] / (main['group'] == 'snow').sum()),
    wheelsAprNovShare=float(main[(main['group'] == 'wheels') & main['m'].between(4, 11)].shape[0] / (main['group'] == 'wheels').sum()),
    dibs=int(notes.get('dibs', 0)), sunk=int(notes.get('sunk', 0)),
    duplicates=int((dup & inwin & sr['kind'].isin(WHEELS + ('snow',))).sum()),
    webApp=float(main['origin'].isin(['Internet', 'Mobile Device']).mean()),
)
summary

## 6. QA
The seasons are real; the classes are not empty; the counts match the portal.

In [ ]:
assert summary['snowWinterShare'] > 0.7, summary['snowWinterShare']
assert summary['wheelsAprNovShare'] > 0.8, summary['wheelsAprNovShare']
assert summary['wheels12'] > summary['snow12'], 'the headline'
assert blocks['n'].sum() == len(main)
assert sum(sum(v) for grp in monthly for v in monthly[grp].values()) == len(main)
assert (grid.values > 0).all(), grid
log.info('blocks per class (rows: reports low-high; columns: winter share low-high)\n%s', grid)
log.info('class sizes, reports: %s', blocks['tc'].value_counts().sort_index().tolist())
import requests
# one calendar year, counted by the portal itself (a narrower query: the full window times out)
q = {'$select': 'count(*) as n', '$where': "sr_type='E-Scooter Parking Complaint' AND created_date>='2025-01-01' AND created_date<'2026-01-01' AND duplicate=false"}
portal = int(requests.get(f'{socrata.BASE_URL}/{DATASET}.json', params=q, timeout=300).json()[0]['n'])
ours = int(((main['kind'] == 'scooter') & (main['yr'] == 2025)).sum())
log.info('portal count, scooters in 2025: %d; ours (with coordinates) %d', portal, ours)
assert abs(portal - ours) < 0.02 * ours

## 7. Export
Blocks as compact rows: x, y (south-west corner, miles), scooter, divvy, snow, winter share, busiest month, area index, report class, winter class.

In [ ]:
area_names = sorted(blocks['area'].unique())
aidx = {a: i for i, a in enumerate(area_names)}
plate = dict(
    snapshot=path.name, dataset=DATASET, step=STEP, reportBreaks=REPORT_BREAKS, winterBreaks=WINTER_BREAKS,
    types={v: k for k, v in TYPES.items()}, areas=area_names,
    blocks=[[r.x, r.y, r.scooter, r.divvy, r.snow, r.winter, r.peak, aidx[r.area], r.tc, r.wc] for r in blocks.itertuples()],
    monthly=monthly, summary=summary,
    sources=dict(sr311='https://data.cityofchicago.org/d/v6vf-nfxy', areas='https://data.cityofchicago.org/d/igwz-8jzy'),
)
finding = (f'In the last 12 months, Chicago residents reported scooters and Divvy bikes parked in the way '
           f'{summary["wheels12"]:,} times, and sidewalks not cleared of snow {summary["snow12"]:,} times.')
ledger = Ledger(
    source='City of Chicago, 311 Service Requests', dataset_id=DATASET, url=plate['sources']['sr311'],
    pulled=pulled, units='reports per half-mile block; reports per month',
    method=['three request types; duplicates flagged by the city dropped', f'window {START} to {END}'],
    caveats=['reports, not obstacles: where people report, not only where things block the way'],
)
write_plate(OUT / 'plate.json', day=16, ledger=ledger, finding=finding, payload=plate)
print(finding)
print('plate.json', round((OUT / 'plate.json').stat().st_size / 1024), 'KB')